In [ ]:
from datasets import load_datasets
from dotenv import laod_dotenv , find_dotenv
import pinecone
from pinecone import Pinecone , ServerLessSpec
import os
from source_transformers import SentenceTransformer

In [ ]:
fw = laod_dataset("HuggingFaceFW/fineweb" , name = "sample=10BT" ,split = "train" , straming = True)

In [ ]:
fw

In [ ]:
model = SentenceTransformers("all-MiniLM-L6-v2")

In [ ]:
load_dotenv(find_dotenv() , override = True)

In [ ]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY") , environment = os.environ.get("PINECONE_ENV"))

In [ ]:
pc.list_indexes()

In [ ]:
pc.create_index(
    name = "text",
    dimension = model.get_sentence_embedding_dimension(),
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
        
        
    )
)


In [ ]:
index = pc.Indexc(name = "text")

In [ ]:
# define the no of items you want to process
subset_size = 10000

vectors_to_upsert = []
for i, item in enumerate(fw):
    if i >= subset_size:
        break

    text = item['text']
    unique_id = str(item['id'])
    language = item['language']

    embedding = model.encode(text , show_progress_bar = False).tolist()

    metadata = {'language' : language}

    vectors_to_upsert.append((unique_id , embedding , metadata))

batch_size = 1000
for i in range(0 , len(vectors_to_upsert) , batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors = batch)

print("share of data upserted to Pinecone index.")